# 08 — Final models on the test set

Each member's final model, fitted on train and scored once on the shared test set by AUC.

> **TODO (Oscar):** a line or two on what this notebook is for and the rule that nothing is tuned here.

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

ROOT = Path.cwd().parent                  # run from report/
PROCESSED = ROOT / "data" / "processed"
TARGET = "default payment next month"

train = pd.read_csv(PROCESSED / "taiwan-train.csv", index_col="ID")
test  = pd.read_csv(PROCESSED / "taiwan-test.csv", index_col="ID")
y_train, y_test = train[TARGET], test[TARGET]

results = {}                              # model name -> test AUC

def test_auc(name, p_test):
    """Score one model's test-set probabilities of default."""
    results[name] = roc_auc_score(y_test, p_test)
    print(f"{name:<30} test AUC {results[name]:.4f}")

print(f"train {len(train):,}   test {len(test):,} customers, {y_test.mean():.2%} defaulted")

train 17,997   test 5,999 customers, 22.12% defaulted


## 1. Logistic regression (Oscar)

In [2]:
sys.path.append(str(ROOT / "Oscar"))
import prep                               # Oscar/prep.py: fixes 1-4 + ratio features

X_train = prep.prepare(train.drop(columns=TARGET), ratios=True)
X_test  = prep.prepare(test.drop(columns=TARGET), ratios=True)

# final model from 05: L2, C = 0.0316 (tuned on CV log loss)
logistic = make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.0316, l1_ratio=0, solver="liblinear", max_iter=1000))
logistic.fit(X_train, y_train);

In [3]:
test_auc("Logistic regression", logistic.predict_proba(X_test)[:, 1])

Logistic regression            test AUC 0.7872


## 2. SVM (Oscar)

In [4]:
# final model from 06: linear SVM, squared hinge, C = 1e-4, Platt scaling for probabilities
# same features as the logistic regression above
svm = make_pipeline(StandardScaler(),
                    CalibratedClassifierCV(LinearSVC(C=1e-4, dual=True, max_iter=20000, random_state=2026),
                                           method="sigmoid", cv=5))
svm.fit(X_train, y_train);

In [5]:
test_auc("SVM", svm.predict_proba(X_test)[:, 1])

SVM                            test AUC 0.7903


## 3. Logit hazard (Alex)

In [6]:
# Alex/modelling/prep.py builds the person-month table; loaded under another name, as Oscar's prep is already imported
spec = importlib.util.spec_from_file_location("hazard_prep", ROOT / "Alex" / "modelling" / "prep.py")
hazard_prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hazard_prep)

pm = hazard_prep.person_months({"train": train, "test": test})   # one row per customer per month, June to October
FEATURES = hazard_prep.features(pm["train"])


def fit_hazard(rows):
    return sm.GLM(rows["event"], sm.add_constant(rows[FEATURES]), family=sm.families.Binomial()).fit()


def october_hazard(model, rows):
    """The hazard in October, i.e. the probability of default, for each customer."""
    october = hazard_prep.october(rows)
    return model.predict(sm.add_constant(october[FEATURES], has_constant="add"))


# final model from 07: logit hazard, with Platt scaling fitted on out-of-fold training predictions
hazard = fit_hazard(pm["train"])
folds = GroupKFold(n_splits=10, shuffle=True, random_state=2026).split(pm["train"], groups=pm["train"].index)
out_of_fold = pd.concat([october_hazard(fit_hazard(pm["train"].iloc[fit_idx]), pm["train"].iloc[score_idx])
                         for fit_idx, score_idx in folds])
log_odds = lambda p: np.log(p / (1 - p)).to_frame("log-odds")
platt = LogisticRegression().fit(log_odds(out_of_fold), y_train.loc[out_of_fold.index])

In [7]:
test_auc("Logit hazard", platt.predict_proba(log_odds(october_hazard(hazard, pm["test"])))[:, 1])

Logit hazard                   test AUC 0.7504


## Results

In [8]:
pd.Series(results, name="test AUC").sort_values(ascending=False).round(4)

SVM                    0.7903
Logistic regression    0.7872
Logit hazard           0.7504
Name: test AUC, dtype: float64